# International Student Enrollment and S&A Funding Research

## Purpose

Investigate how international student enrollment relates to Seattle Central's
overall enrollment and Services & Activities (S&A) funding.

This analysis distinguishes between:

- international students generally;
- International Contract students;
- state-funded international students; and
- resident/nonresident state-funded students.

## Research Questions

1. How many international students attended Seattle Central in each dashboard year?
2. Are international students included in the annual all-source headcount?
3. How many international students are coded as International Contract?
4. How many are recorded as state-funded?
5. How does tuition/revenue treatment differ between these groups?
6. Where is International Contract revenue recorded?
7. Does any International Contract revenue ultimately transfer to Fund 522?
8. How should international students be represented when interpreting S&A funding?

## Dashboard Years

- 2023–24
- 2024–25
- 2025–26

## Important Distinction

"International student" and "International Contract student" are not
necessarily equivalent categories.

The analysis must not assume that all international students use the
International Contract funding classification.

In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

INTERNATIONAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "manual"
    / "international_enrollment_context.csv"
)

international = pd.read_csv(
    INTERNATIONAL_PATH
)

print("INTERNATIONAL ENROLLMENT CONTEXT")
print("=" * 50)

print(f"Rows: {len(international):,}")
print(f"Columns: {len(international.columns):,}")

display(international)

INTERNATIONAL ENROLLMENT CONTEXT
Rows: 3
Columns: 9


,academic_year,total_headcount,international_all_funds,international_state_fund,international_contract_fund,international_contract_program_all,international_contract_program_contract,status,notes
0,2023-24,12088,1058,32,907,911,911,final,Annual headcount; Seattle Central/SVI
1,2024-25,12378,1132,26,990,994,994,final,Annual headcount; Seattle Central/SVI
2,2025-26,11672,1035,11,924,927,927,provisional,Most recent academic year; SBCTC notes final d...


In [3]:
# =========================================================
# INTERNATIONAL ENROLLMENT VALIDATION
# =========================================================

print("INTERNATIONAL ENROLLMENT VALIDATION")
print("=" * 50)


# 1. EXPECTED ACADEMIC YEARS

expected_years = {
    "2023-24",
    "2024-25",
    "2025-26",
}

actual_years = set(international["academic_year"])

years_correct = actual_years == expected_years

print(f"Academic years correct: {years_correct}")


# 2. DUPLICATE RECORDS

duplicate_years = international.duplicated(
    subset=["academic_year"]
).sum()

print(f"Duplicate years: {duplicate_years}")


# 3. MISSING REQUIRED VALUES

required_columns = [
    "academic_year",
    "total_headcount",
    "international_all_funds",
    "international_state_fund",
    "international_contract_fund",
    "international_contract_program_all",
    "international_contract_program_contract",
    "status",
]

missing_values = international[
    required_columns
].isna().sum()

print("\nMissing required values:")
print(missing_values)


# 4. VALIDATE STUDENT COUNTS

count_columns = [
    "total_headcount",
    "international_all_funds",
    "international_state_fund",
    "international_contract_fund",
    "international_contract_program_all",
    "international_contract_program_contract",
]

counts = international[count_columns].apply(
    pd.to_numeric, errors="coerce"
)

counts_valid = (
    counts.notna().all().all()
    and counts.ge(0).all().all()
    and counts.mod(1).eq(0).all().all()
)

population_check = (
    international["international_all_funds"]
    <= international["total_headcount"]
).all()

fund_filter_check = (
    counts["international_state_fund"]
    .le(counts["international_all_funds"]).all()
    and
    counts["international_contract_fund"]
    .le(counts["international_all_funds"]).all()
)

print(f"\nStudent counts valid: {counts_valid}")
print(f"International <= total: {population_check}")
print(f"Funding filters within total: {fund_filter_check}")


# 5. CONTRACT PROGRAM CONSISTENCY

contract_match = (
    international["international_contract_program_all"]
    ==
    international["international_contract_program_contract"]
).all()

print(f"Contract program counts match: {contract_match}")


# 6. SOURCE REPORTING DIFFERENCE
# Record this discrepancy; do not force equality.

international["contract_group_difference"] = (
    international["international_contract_program_all"]
    -
    international["international_contract_fund"]
)

print("\nContract reporting differences:")

display(
    international[
        [
            "academic_year",
            "contract_group_difference",
        ]
    ]
)


# 7. DATA STATUS

expected_status = {
    "2023-24": "final",
    "2024-25": "final",
    "2025-26": "provisional",
}

status_check = all(
    international.loc[
        international["academic_year"] == year,
        "status",
    ].iloc[0] == status
    for year, status in expected_status.items()
)

print(f"\nReporting statuses correct: {status_check}")


# 8. CROSS-CHECK MAIN ENROLLMENT DATA

enrollment_reference = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "manual"
    / "enrollment_context.csv"
)

crosscheck = international[
    ["academic_year", "total_headcount"]
].merge(
    enrollment_reference[
        ["academic_year", "total_students"]
    ],
    on="academic_year",
    how="left",
    validate="one_to_one",
)

enrollment_match = (
    crosscheck["total_headcount"]
    ==
    crosscheck["total_students"]
).all()

print(f"Matches main enrollment data: {enrollment_match}")


# FINAL VALIDATION

validation_passed = all([
    years_correct,
    duplicate_years == 0,
    missing_values.sum() == 0,
    counts_valid,
    population_check,
    fund_filter_check,
    contract_match,
    status_check,
    enrollment_match,
])

print("\n" + "=" * 50)

print(
    "VALIDATION PASSED"
    if validation_passed
    else "VALIDATION FAILED"
)

INTERNATIONAL ENROLLMENT VALIDATION
Academic years correct: True
Duplicate years: 0

Missing required values:
academic_year                              0
total_headcount                            0
international_all_funds                    0
international_state_fund                   0
international_contract_fund                0
international_contract_program_all         0
international_contract_program_contract    0
status                                     0
dtype: int64

Student counts valid: True
International <= total: True
Funding filters within total: True
Contract program counts match: True

Contract reporting differences:


,academic_year,contract_group_difference
0,2023-24,4
1,2024-25,4
2,2025-26,3



Reporting statuses correct: True
Matches main enrollment data: True

VALIDATION PASSED


In [4]:
# =========================================================
# INTERNATIONAL ENROLLMENT ANALYSIS
# =========================================================

international_summary = (
    international
    .sort_values("academic_year")
    .copy()
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 1. INTERNATIONAL SHARE OF TOTAL ENROLLMENT
# ---------------------------------------------------------

international_summary["international_share_pct"] = (
    international_summary["international_all_funds"]
    / international_summary["total_headcount"]
    * 100
)


# ---------------------------------------------------------
# 2. FUNDING CATEGORY SHARES
# ---------------------------------------------------------

# These describe filtered populations, not exclusive groups.
# A student may appear under multiple funding categories.

international_summary["intl_state_share_pct"] = (
    international_summary["international_state_fund"]
    / international_summary["international_all_funds"]
    * 100
)

international_summary["intl_contract_share_pct"] = (
    international_summary["international_contract_fund"]
    / international_summary["international_all_funds"]
    * 100
)


# ---------------------------------------------------------
# 3. INTERNATIONAL CONTRACT SHARE OF TOTAL
# ---------------------------------------------------------

international_summary["contract_program_share_pct"] = (
    international_summary["international_contract_program_all"]
    / international_summary["total_headcount"]
    * 100
)


# ---------------------------------------------------------
# 4. YEAR-OVER-YEAR ENROLLMENT CHANGES
# ---------------------------------------------------------

international_summary["intl_yoy_change"] = (
    international_summary["international_all_funds"]
    .diff()
)

international_summary["intl_yoy_pct"] = (
    international_summary["international_all_funds"]
    .pct_change(fill_method=None)
    * 100
)

international_summary["contract_yoy_change"] = (
    international_summary["international_contract_program_all"]
    .diff()
)

international_summary["contract_yoy_pct"] = (
    international_summary["international_contract_program_all"]
    .pct_change(fill_method=None)
    * 100
)


# ---------------------------------------------------------
# 5. DISPLAY RESULTS
# ---------------------------------------------------------

percentage_columns = [
    "international_share_pct",
    "intl_state_share_pct",
    "intl_contract_share_pct",
    "contract_program_share_pct",
    "intl_yoy_pct",
    "contract_yoy_pct",
]

international_summary[percentage_columns] = (
    international_summary[percentage_columns]
    .round(2)
)


print("INTERNATIONAL ENROLLMENT SHARES")
print("=" * 50)

display(
    international_summary[
        [
            "academic_year",
            "international_all_funds",
            "international_share_pct",
            "intl_state_share_pct",
            "intl_contract_share_pct",
            "contract_program_share_pct",
        ]
    ]
)


print("\nYEAR-OVER-YEAR ENROLLMENT CHANGES")
print("=" * 50)

display(
    international_summary[
        [
            "academic_year",
            "international_all_funds",
            "intl_yoy_change",
            "intl_yoy_pct",
            "international_contract_program_all",
            "contract_yoy_change",
            "contract_yoy_pct",
            "status",
        ]
    ]
)

INTERNATIONAL ENROLLMENT SHARES


,academic_year,international_all_funds,international_share_pct,intl_state_share_pct,intl_contract_share_pct,contract_program_share_pct
0,2023-24,1058,8.75,3.02,85.73,7.54
1,2024-25,1132,9.15,2.30,87.46,8.03
2,2025-26,1035,8.87,1.06,89.28,7.94



YEAR-OVER-YEAR ENROLLMENT CHANGES


,academic_year,international_all_funds,intl_yoy_change,intl_yoy_pct,international_contract_program_all,contract_yoy_change,contract_yoy_pct,status
0,2023-24,1058,NaN,NaN,911,NaN,NaN,final
1,2024-25,1132,74.0,6.99,994,83.0,9.11,final
2,2025-26,1035,-97.0,-8.57,927,-67.0,-6.74,provisional


In [5]:
# =========================================================
# FULL-PERIOD INTERNATIONAL ENROLLMENT CHANGE
# =========================================================

start = international_summary.loc[
    international_summary["academic_year"] == "2023-24"
].iloc[0]

end = international_summary.loc[
    international_summary["academic_year"] == "2025-26"
].iloc[0]


# 1. OVERALL INTERNATIONAL ENROLLMENT

intl_change = (
    end["international_all_funds"]
    - start["international_all_funds"]
)

intl_change_pct = (
    intl_change
    / start["international_all_funds"]
    * 100
)


# 2. INTERNATIONAL CONTRACT ENROLLMENT

contract_change = (
    end["international_contract_program_all"]
    - start["international_contract_program_all"]
)

contract_change_pct = (
    contract_change
    / start["international_contract_program_all"]
    * 100
)


# 3. CHANGE IN SHARE OF TOTAL ENROLLMENT

share_change_pp = (
    end["international_share_pct"]
    - start["international_share_pct"]
)


# 4. DISPLAY RESULTS

print("FULL-PERIOD ENROLLMENT CHANGE")
print("=" * 50)

print("International students:")
print(f"Net change: {intl_change:+,.0f}")
print(f"Percent change: {intl_change_pct:+.2f}%")

print("\nInternational Contract students:")
print(f"Net change: {contract_change:+,.0f}")
print(f"Percent change: {contract_change_pct:+.2f}%")

print("\nInternational share of total enrollment:")
print(f"Change: {share_change_pp:+.2f} percentage points")

FULL-PERIOD ENROLLMENT CHANGE
International students:
Net change: -23
Percent change: -2.17%

International Contract students:
Net change: +16
Percent change: +1.76%

International share of total enrollment:
Change: +0.12 percentage points


In [7]:
# =========================================================
# FINAL VALIDATION AND EXPORT
# =========================================================

print("FINAL INTERNATIONAL ENROLLMENT CHECK")
print("=" * 50)

# 1. Check expected years and data completeness

expected_years = {
    "2023-24",
    "2024-25",
    "2025-26",
}

required_columns = [
    "academic_year",
    "total_headcount",
    "international_all_funds",
    "international_state_fund",
    "international_contract_fund",
    "international_contract_program_all",
    "international_contract_program_contract",
    "international_share_pct",
    "intl_state_share_pct",
    "intl_contract_share_pct",
    "contract_program_share_pct",
    "status",
]

years_valid = (
    set(international_summary["academic_year"])
    == expected_years
)

no_duplicates = (
    not international_summary["academic_year"]
    .duplicated().any()
)

no_missing = (
    international_summary[required_columns]
    .notna().all().all()
)

differences_valid = (
    international_summary["contract_group_difference"]
    .tolist() == [4, 4, 3]
)

final_check = all([
    validation_passed,
    years_valid,
    no_duplicates,
    no_missing,
    differences_valid,
])

print(f"Prior validation passed: {validation_passed}")
print(f"Years valid: {years_valid}")
print(f"No duplicates: {no_duplicates}")
print(f"No missing required data: {no_missing}")
print(f"Reporting differences preserved: {differences_valid}")


# =========================================================
# EXPORT PROCESSED DATA
# =========================================================

if final_check:

    output = international_summary.copy()

    # Convert year-over-year changes to nullable integers
    for col in ["intl_yoy_change", "contract_yoy_change"]:
        output[col] = output[col].astype("Int64")

    OUTPUT_PATH = (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "international_enrollment_summary.csv"
    )

    output.to_csv(
        OUTPUT_PATH,
        index=False
    )

    print("\nALL CHECKS PASSED")
    print(f"Saved: {OUTPUT_PATH}")
    print(f"Rows exported: {len(output)}")

else:
    print("\nVALIDATION FAILED — export skipped")
    
    


FINAL INTERNATIONAL ENROLLMENT CHECK
Prior validation passed: True
Years valid: True
No duplicates: True
No missing required data: True
Reporting differences preserved: True

ALL CHECKS PASSED
Saved: /Users/mainguyen/GitHub/seattle-central-sa-budget-dashboard/data/processed/international_enrollment_summary.csv
Rows exported: 3



## International Contract Revenue and S&A Accounting

### Research Objective

Determine whether International Contract payments contribute to Seattle Central's S&A budget, directly or through internal transfers.

### Verified Statewide Accounting Rules

**Standard S&A tuition**
- 96.5%: Fund 522, Class 264 (S&A)
- 3.5%: Fund 860, Class 279 (institutional financial aid)

**International Contract**
- Fee-in-lieu of tuition
- Initial accounting: Fund 146
- Revenue account: 4021065
- SBCTC remittance: 2.5% beginning FY2024

### Seattle Colleges-Specific Findings

Seattle Colleges identifies Fund 146 (grants/contracts) and Fund 522 (student activity fees) separately.

Historical Seattle Central student-government records show that collecting additional S&A fees from international students was considered but not established as an approved policy in those discussions.

### Unresolved Questions

1. Does International Contract revenue ever transfer to Fund 522?
2. Are there direct contributions from International Programs to student activities instead?
3. Was the proposed international S&A policy subsequently approved?
4. Can annual revenue or transfer amounts be verified from financial records?

### Research Limitations

- Enrollment figures do not establish revenue amounts.
- Fund classifications establish initial accounting treatment, not every subsequent transaction.
- Historical meeting minutes do not establish current policy.
- No confirmed International Contract-to-S&A transfer has yet been identified.

### Official Sources

- https://www.sbctc.edu/colleges-staff/policies-rules/policy-manual/chapter-5.aspx
- https://www.sbctc.edu/colleges-staff/programs-services/accounting-business/clam/student-financials/student-financials-processes.aspx
- https://www.sbctc.edu/colleges-staff/programs-services/accounting-business/rs-ic-revenue-remittance.aspx
- https://www.seattlecolleges.edu/about/policies-and-procedures/pro605?companionId=pro&docID=605&hasboth=1


In [8]:
# =========================================================
# INTERNATIONAL S&A FUNDING — SOURCE EVIDENCE LOG
# =========================================================

evidence_records = [
    {
        "year": "Current policy",
        "source": "SBCTC Chart of Accounts",
        "topic": "Fund 522",
        "finding": "Associated Students fund records approved "
                   "student-activity revenue and expenditures.",
        "status": "Verified statewide",
        "source_url": "https://www.sbctc.edu/colleges-staff/programs-services/accounting-business/clam/chart-of-accounts/funds",
    },
    {
        "year": "Current policy",
        "source": "SBCTC Chart of Accounts",
        "topic": "Fund 146",
        "finding": "Contracts fund includes International Contract, "
                   "Running Start, and other contract programs.",
        "status": "Verified statewide",
        "source_url": "https://www.sbctc.edu/colleges-staff/programs-services/accounting-business/clam/chart-of-accounts/funds",
    },
    {
        "year": "Current policy",
        "source": "SBCTC Student Financials",
        "topic": "S&A revenue allocation",
        "finding": "Standard S&A tuition: 96.5% to Fund 522 "
                   "and 3.5% to Fund 860.",
        "status": "Verified statewide",
        "source_url": "https://www.sbctc.edu/colleges-staff/programs-services/accounting-business/clam/student-financials/student-financials-processes.aspx",
    },
    {
        "year": "Current policy",
        "source": "SBCTC Policy Chapter 5",
        "topic": "International Contract tuition",
        "finding": "Fee-in-lieu revenue goes to the "
                   "Grants and Contracts account.",
        "status": "Verified statewide",
        "source_url": "https://www.sbctc.edu/colleges-staff/policies-rules/policy-manual/chapter-5",
    },
    {
        "year": "FY2024 onward",
        "source": "SBCTC Remittance Guidance",
        "topic": "International revenue coding",
        "finding": "Fund 146; revenue account 4021065. "
                   "SBCTC remittance rate is 2.5%.",
        "status": "Verified statewide",
        "source_url": "https://www.sbctc.edu/colleges-staff/programs-services/accounting-business/rs-ic-revenue-remittance.aspx",
    },
    {
        "year": "FY2025",
        "source": "Seattle Colleges Budget Actions",
        "topic": "District fund distinctions",
        "finding": "District budget guidance identifies "
                   "international revenues and student "
                   "activity fees as distinct categories.",
        "status": "Verified district document",
        "source_url": "https://www.seattlecolleges.edu/administration/budget-info/fy2025-budget-info/fy2025-budget-actions-and-decisions",
    },
    {
        "year": "2021",
        "source": "Seattle Central College Council",
        "topic": "Historical international S&A treatment",
        "finding": "Minutes describe historical noncollection "
                   "of S&A fees and prior International "
                   "Programs financial support.",
        "status": "Historical evidence",
        "source_url": "https://seattlecentral.edu/sites/default/files/inline-files/College%20Council%20Minutes%2003.16.21.pdf",
    },
    {
        "year": "2023",
        "source": "Seattle Central College Council",
        "topic": "Potential fee policy change",
        "finding": "Discussion of a districtwide task force "
                   "to investigate charging international "
                   "and Running Start students S&A fees.",
        "status": "Historical proposal",
        "source_url": "https://seattlecentral.edu/sites/default/files/2023-11/June_2023_CC_Minutes.pdf",
    },
    {
        "year": "2024",
        "source": "Seattle Central College Council",
        "topic": "Continued discussion",
        "finding": "Minutes still describe adding these "
                   "student fees as being discussed.",
        "status": "Historical proposal",
        "source_url": "https://seattlecentral.edu/sites/default/files/2024-03/College%20Council%20Minutes%201.18.pdf",
    },
]

fund_evidence = pd.DataFrame(evidence_records)

assert fund_evidence["source_url"].notna().all()
assert not fund_evidence[["source", "topic"]].duplicated().any()

print("S&A FUNDING EVIDENCE LOG")
print("=" * 50)
print(f"Evidence records: {len(fund_evidence)}")

display(fund_evidence)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "manual"
    / "sa_fund_evidence_log.csv"
)

fund_evidence.to_csv(OUTPUT_PATH, index=False)

print(f"\nSaved: {OUTPUT_PATH}")

S&A FUNDING EVIDENCE LOG
Evidence records: 9


,year,source,topic,finding,status,source_url
0,Current policy,SBCTC Chart of Accounts,Fund 522,Associated Students fund records approved stud...,Verified statewide,https://www.sbctc.edu/colleges-staff/programs-...
1,Current policy,SBCTC Chart of Accounts,Fund 146,Contracts fund includes International Contract...,Verified statewide,https://www.sbctc.edu/colleges-staff/programs-...
2,Current policy,SBCTC Student Financials,S&A revenue allocation,Standard S&A tuition: 96.5% to Fund 522 and 3....,Verified statewide,https://www.sbctc.edu/colleges-staff/programs-...
3,Current policy,SBCTC Policy Chapter 5,International Contract tuition,Fee-in-lieu revenue goes to the Grants and Con...,Verified statewide,https://www.sbctc.edu/colleges-staff/policies-...
4,FY2024 onward,SBCTC Remittance Guidance,International revenue coding,Fund 146; revenue account 4021065. SBCTC remit...,Verified statewide,https://www.sbctc.edu/colleges-staff/programs-...
5,FY2025,Seattle Colleges Budget Actions,District fund distinctions,District budget guidance identifies internatio...,Verified district document,https://www.seattlecolleges.edu/administration...
6,2021,Seattle Central College Council,Historical international S&A treatment,Minutes describe historical noncollection of S...,Historical evidence,https://seattlecentral.edu/sites/default/files...
7,2023,Seattle Central College Council,Potential fee policy change,Discussion of a districtwide task force to inv...,Historical proposal,https://seattlecentral.edu/sites/default/files...
8,2024,Seattle Central College Council,Continued discussion,Minutes still describe adding these student fe...,Historical proposal,https://seattlecentral.edu/sites/default/files...



Saved: /Users/mainguyen/GitHub/seattle-central-sa-budget-dashboard/data/manual/sa_fund_evidence_log.csv


In [9]:
# =========================================================
# ADD FINANCIAL RESEARCH EVIDENCE
# =========================================================

new_records = [
    {
        "year": "2021",
        "source": "Seattle Central ASC Minutes",
        "topic": "International S&A approval status",
        "finding": "ASC reported on April 12, 2021 that "
                   "international S&A had not yet been approved.",
        "status": "Historical evidence",
        "source_url": "https://studentleadership.seattlecentral.edu/sites/studentleadership.seattlecentral.edu/files/inline-files/ASC%20minutes%20April%2012th.pdf",
    },
    {
        "year": "FY2027",
        "source": "Seattle Colleges Spending Freeze Guidance",
        "topic": "Current district fund classifications",
        "finding": "District guidance separately identifies "
                   "grants/contracts Funds 145/146 and "
                   "student activity fees Fund 522.",
        "status": "Verified district document",
        "source_url": "https://www.seattlecolleges.edu/administration/budget-info/fiscal-year-2026-2027-spending-freeze/hiring-and-personnel",
    },
    {
        "year": "FY2025",
        "source": "Seattle Colleges Audited Financial Statements",
        "topic": "Interfund transaction visibility",
        "finding": "Interfund receivables and payables are "
                   "generally eliminated in financial reporting; "
                   "specific fund transfers remain unverified.",
        "status": "Reporting limitation",
        "source_url": "https://www.seattlecolleges.edu/sites/seattlecolleges.southseattle.edu/files/2026-03/FY25%20Seattle%20Colleges%20Audited%20Financial%20Statements.pdf",
    },
]

fund_evidence = pd.concat(
    [
        fund_evidence,
        pd.DataFrame(new_records),
    ],
    ignore_index=True,
)

assert not fund_evidence[
    ["source", "topic"]
].duplicated().any()

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "manual"
    / "sa_fund_evidence_log.csv"
)

fund_evidence.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("UPDATED S&A FUNDING EVIDENCE LOG")
print("=" * 50)

print(f"Total evidence records: {len(fund_evidence)}")

display(fund_evidence.tail(3))

UPDATED S&A FUNDING EVIDENCE LOG
Total evidence records: 12


,year,source,topic,finding,status,source_url
9,2021,Seattle Central ASC Minutes,International S&A approval status,"ASC reported on April 12, 2021 that internatio...",Historical evidence,https://studentleadership.seattlecentral.edu/s...
10,FY2027,Seattle Colleges Spending Freeze Guidance,Current district fund classifications,District guidance separately identifies grants...,Verified district document,https://www.seattlecolleges.edu/administration...
11,FY2025,Seattle Colleges Audited Financial Statements,Interfund transaction visibility,Interfund receivables and payables are general...,Reporting limitation,https://www.seattlecolleges.edu/sites/seattlec...


In [10]:
# =========================================================
# FINANCIAL RESEARCH — OPEN QUESTIONS
# =========================================================

research_questions = [
    {
        "question": "Was an international S&A fee approved?",
        "period": "2024-2026",
        "evidence_needed": (
            "Board resolution, approved fee schedule, "
            "or formal policy amendment"
        ),
        "source_to_check": "Board of Trustees records",
        "status": "Unresolved",
    },
    {
        "question": "Does Fund 146 transfer money to Fund 522?",
        "period": "FY2024-FY2026",
        "evidence_needed": (
            "Interfund transfer journals or general "
            "ledger transaction details"
        ),
        "source_to_check": "Seattle Colleges Finance",
        "status": "Unresolved",
    },
    {
        "question": "Does International Programs support S&A?",
        "period": "FY2024-FY2026",
        "evidence_needed": (
            "Direct payments, subsidies, "
            "reimbursements, or agreements"
        ),
        "source_to_check": (
            "International Programs / Student Leadership"
        ),
        "status": "Unresolved",
    },
    {
        "question": "How much S&A revenue was collected?",
        "period": "FY2024-FY2026",
        "evidence_needed": (
            "Seattle Central Fund 522 revenue report, "
            "including revenue categories"
        ),
        "source_to_check": "Seattle Central Business Office",
        "status": "Unresolved",
    },
]

fund_open_questions = pd.DataFrame(research_questions)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "manual"
    / "sa_fund_open_questions.csv"
)

fund_open_questions.to_csv(
    OUTPUT_PATH,
    index=False
)

print("FINANCIAL RESEARCH TRACKER")
print("=" * 50)

print(f"Open questions: {len(fund_open_questions)}")

display(fund_open_questions)

FINANCIAL RESEARCH TRACKER
Open questions: 4


,question,period,evidence_needed,source_to_check,status
0,Was an international S&A fee approved?,2024-2026,"Board resolution, approved fee schedule, or fo...",Board of Trustees records,Unresolved
1,Does Fund 146 transfer money to Fund 522?,FY2024-FY2026,Interfund transfer journals or general ledger ...,Seattle Colleges Finance,Unresolved
2,Does International Programs support S&A?,FY2024-FY2026,"Direct payments, subsidies, reimbursements, or...",International Programs / Student Leadership,Unresolved
3,How much S&A revenue was collected?,FY2024-FY2026,"Seattle Central Fund 522 revenue report, inclu...",Seattle Central Business Office,Unresolved


In [11]:
# =========================================================
# ADD INTERNATIONAL FEE EVIDENCE
# =========================================================

BOARD_PACKET = (
    "https://seattlecolleges.community.diligentoneplatform.com/"
    "document/6589/?lastModified=638799069088370000"
)

CURRENT_FEES = (
    "https://seattlecentral.edu/tuition-and-fees/fee-schedule"
)

new_evidence = pd.DataFrame([
    {
        "year": "2025",
        "source": "April 2025 Board Fee Packet",
        "topic": "International Activity Fee",
        "finding": (
            "Page 69 lists SB: Int'l Activity Fee as "
            "variable. Its destination fund is not stated."
        ),
        "status": "Verified fee listing",
        "source_url": BOARD_PACKET,
    },
    {
        "year": "Current listing",
        "source": "Seattle Central Fee Schedule",
        "topic": "International Activity Fee",
        "finding": (
            "The activity fee remains listed as variable. "
            "No Fund 522 connection is specified."
        ),
        "status": "Verified fee listing",
        "source_url": CURRENT_FEES,
    },
    {
        "year": "Current listing",
        "source": "Seattle Central Fee Schedule",
        "topic": "Recreation Center - No S&A",
        "finding": (
            "The $60 fee is optional for students who "
            "do not automatically pay for MAC use. "
            "It is not identified as international-only."
        ),
        "status": "Verified fee listing",
        "source_url": CURRENT_FEES,
    },
])

LOG_PATH = (
    PROJECT_ROOT
    / "data"
    / "manual"
    / "sa_fund_evidence_log.csv"
)

# Reload the saved log so reruns do not duplicate records.
fund_evidence = pd.read_csv(LOG_PATH)

fund_evidence = (
    pd.concat(
        [fund_evidence, new_evidence],
        ignore_index=True
    )
    .drop_duplicates(
        subset=["source", "topic"],
        keep="last"
    )
    .reset_index(drop=True)
)

fund_evidence.to_csv(LOG_PATH, index=False)

print("UPDATED FUND EVIDENCE")
print("=" * 50)
print(f"Total evidence records: {len(fund_evidence)}")

display(fund_evidence.tail(3))

UPDATED FUND EVIDENCE
Total evidence records: 15


,year,source,topic,finding,status,source_url
12,2025,April 2025 Board Fee Packet,International Activity Fee,Page 69 lists SB: Int'l Activity Fee as variab...,Verified fee listing,https://seattlecolleges.community.diligentonep...
13,Current listing,Seattle Central Fee Schedule,International Activity Fee,The activity fee remains listed as variable. N...,Verified fee listing,https://seattlecentral.edu/tuition-and-fees/fe...
14,Current listing,Seattle Central Fee Schedule,Recreation Center - No S&A,The $60 fee is optional for students who do no...,Verified fee listing,https://seattlecentral.edu/tuition-and-fees/fe...


In [12]:
# =========================================================
# INTERNATIONAL FEES — REFERENCE DATA
# =========================================================

fee_items = [
    ("SB", "International Activity", None, None,
     "variable", "S&A investigation"),

    ("SV", "International Student Services", 200, 200,
     "quarterly", "Supporting context"),

    ("SEVP", "SEVP Reporting", 20, 22,
     "term", "Supporting context"),

    ("", "International Insurance", 451.71, 488.58,
     "term", "Supporting context"),

    ("", "International Orientation", 50, 50,
     "incidental", "Supporting context"),

    ("", "District Accounting Administration", 40, 40,
     "incidental", "Supporting context"),

    ("DF", "Recreation Center - No S&A", 60, 60,
     "quarterly", "S&A investigation"),
]

records = []

for code, name, old_amount, new_amount, basis, focus in fee_items:

    for snapshot, amount, url, page in [
        ("2025_board_packet", old_amount, BOARD_PACKET, 69),
        ("current_website", new_amount, CURRENT_FEES, None),
    ]:

        records.append({
            "source_snapshot": snapshot,
            "fee_code": code,
            "fee_name": name,
            "published_amount_usd": amount,
            "amount_type": (
                "variable" if amount is None else "fixed"
            ),
            "reference_basis": basis,
            "research_focus": focus,
            "destination_fund": "unverified",
            "source_page": page,
            "source_url": url,
        })

international_fees = pd.DataFrame(records)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "manual"
    / "international_fee_reference.csv"
)

assert not international_fees[
    ["source_snapshot", "fee_name"]
].duplicated().any()

international_fees.to_csv(OUTPUT_PATH, index=False)

print("INTERNATIONAL FEE REFERENCE")
print("=" * 50)
print(f"Rows: {len(international_fees)}")
print(f"Saved: {OUTPUT_PATH}")

display(international_fees)

INTERNATIONAL FEE REFERENCE
Rows: 14
Saved: /Users/mainguyen/GitHub/seattle-central-sa-budget-dashboard/data/manual/international_fee_reference.csv


,source_snapshot,fee_code,fee_name,published_amount_usd,amount_type,reference_basis,research_focus,destination_fund,source_page,source_url
0,2025_board_packet,SB,International Activity,NaN,variable,variable,S&A investigation,unverified,69.0,https://seattlecolleges.community.diligentonep...
1,current_website,SB,International Activity,NaN,variable,variable,S&A investigation,unverified,NaN,https://seattlecentral.edu/tuition-and-fees/fe...
2,2025_board_packet,SV,International Student Services,200.00,fixed,quarterly,Supporting context,unverified,69.0,https://seattlecolleges.community.diligentonep...
3,current_website,SV,International Student Services,200.00,fixed,quarterly,Supporting context,unverified,NaN,https://seattlecentral.edu/tuition-and-fees/fe...
4,2025_board_packet,SEVP,SEVP Reporting,20.00,fixed,term,Supporting context,unverified,69.0,https://seattlecolleges.community.diligentonep...
5,current_website,SEVP,SEVP Reporting,22.00,fixed,term,Supporting context,unverified,NaN,https://seattlecentral.edu/tuition-and-fees/fe...
6,2025_board_packet,,International Insurance,451.71,fixed,term,Supporting context,unverified,69.0,https://seattlecolleges.community.diligentonep...
7,current_website,,International Insurance,488.58,fixed,term,Supporting context,unverified,NaN,https://seattlecentral.edu/tuition-and-fees/fe...
8,2025_board_packet,,International Orientation,50.00,fixed,incidental,Supporting context,unverified,69.0,https://seattlecolleges.community.diligentonep...
9,current_website,,International Orientation,50.00,fixed,incidental,Supporting context,unverified,NaN,https://seattlecentral.edu/tuition-and-fees/fe...


In [13]:
# =========================================================
# ACCOUNTING CONFIGURATION EVIDENCE
# =========================================================

new_records = [
    {
        "year": "2025",
        "source": "April 2025 Board Fee Packet",
        "topic": "SB fee classification",
        "finding": (
            "Page 69 classifies SB: Int'l Activity Fee "
            "as MISC / incidental / variable. "
            "Assessment and destination fund are unverified."
        ),
        "status": "Verified fee classification",
        "source_url": (
            "https://seattlecolleges.community."
            "diligentoneplatform.com/document/6589/"
            "?lastModified=638799069088370000"
        ),
    },
    {
        "year": "Current accounting guidance",
        "source": "SBCTC Student Financials Item Types",
        "topic": "Fee general-ledger configuration",
        "finding": (
            "Student Financials Item Types determine "
            "accounting destinations. Seattle Central's "
            "specific SB fee Item Type mapping is unverified."
        ),
        "status": "Verified statewide / local mapping unresolved",
        "source_url": (
            "https://www.sbctc.edu/colleges-staff/"
            "programs-services/accounting-business/"
            "clam/student-financials/"
            "student-financials-item-types.aspx"
        ),
    },
]

LOG_PATH = (
    PROJECT_ROOT / "data" / "manual"
    / "sa_fund_evidence_log.csv"
)

fund_evidence = pd.read_csv(LOG_PATH)

fund_evidence = (
    pd.concat(
        [fund_evidence, pd.DataFrame(new_records)],
        ignore_index=True
    )
    .drop_duplicates(
        subset=["source", "topic"],
        keep="last"
    )
    .reset_index(drop=True)
)

fund_evidence.to_csv(LOG_PATH, index=False)

print(f"Evidence records: {len(fund_evidence)}")
display(fund_evidence.tail(2))

Evidence records: 17


,year,source,topic,finding,status,source_url
15,2025,April 2025 Board Fee Packet,SB fee classification,Page 69 classifies SB: Int'l Activity Fee as M...,Verified fee classification,https://seattlecolleges.community.diligentonep...
16,Current accounting guidance,SBCTC Student Financials Item Types,Fee general-ledger configuration,Student Financials Item Types determine accoun...,Verified statewide / local mapping unresolved,https://www.sbctc.edu/colleges-staff/programs-...


In [14]:
# =========================================================
# ADD SEATTLE CENTRAL ACCOUNTING EVIDENCE
# =========================================================

new_records = [
    {
        "year": "2023",
        "source": "Seattle Central ASC Procedural Manual",
        "topic": "Fund 522 revenue sources",
        "finding": (
            "Fund 522 supports approved student programs. "
            "Revenue may include S&A fees, sponsored "
            "activities, and individual contributions."
        ),
        "status": "Verified 2023 college procedure",
        "source_url": (
            "https://studentleadership.seattlecentral.edu/"
            "sites/studentleadership.seattlecentral.edu/"
            "files/2023-09/procedural-manual-2023.pdf"
        ),
    },
    {
        "year": "2023",
        "source": "Seattle Central ASC Procedural Manual",
        "topic": "Financial reporting responsibility",
        "finding": (
            "Director of Business Operations maintains "
            "S&A financial transactions and provides "
            "periodic reports to designated college "
            "and student leadership officials."
        ),
        "status": "Verified 2023 college procedure",
        "source_url": (
            "https://studentleadership.seattlecentral.edu/"
            "sites/studentleadership.seattlecentral.edu/"
            "files/2023-09/procedural-manual-2023.pdf"
        ),
    },
    {
        "year": "Current accounting guidance",
        "source": "SBCTC Student Financials Item Types",
        "topic": "Miscellaneous fee fund destinations",
        "finding": (
            "Miscellaneous fees commonly use Fund 148, "
            "but accounting destinations vary. "
            "The SB fee mapping remains unverified."
        ),
        "status": "General rule / local mapping unresolved",
        "source_url": (
            "https://www.sbctc.edu/colleges-staff/"
            "programs-services/accounting-business/"
            "clam/student-financials/"
            "student-financials-item-types.aspx"
        ),
    },
]

LOG_PATH = (
    PROJECT_ROOT / "data" / "manual"
    / "sa_fund_evidence_log.csv"
)

fund_evidence = pd.read_csv(LOG_PATH)

fund_evidence = (
    pd.concat(
        [fund_evidence, pd.DataFrame(new_records)],
        ignore_index=True
    )
    .drop_duplicates(
        subset=["source", "topic"],
        keep="last"
    )
    .reset_index(drop=True)
)

fund_evidence.to_csv(LOG_PATH, index=False)

print(f"Total evidence records: {len(fund_evidence)}")

display(fund_evidence.tail(3))

Total evidence records: 20


,year,source,topic,finding,status,source_url
17,2023,Seattle Central ASC Procedural Manual,Fund 522 revenue sources,Fund 522 supports approved student programs. R...,Verified 2023 college procedure,https://studentleadership.seattlecentral.edu/s...
18,2023,Seattle Central ASC Procedural Manual,Financial reporting responsibility,Director of Business Operations maintains S&A ...,Verified 2023 college procedure,https://studentleadership.seattlecentral.edu/s...
19,Current accounting guidance,SBCTC Student Financials Item Types,Miscellaneous fee fund destinations,"Miscellaneous fees commonly use Fund 148, but ...",General rule / local mapping unresolved,https://www.sbctc.edu/colleges-staff/programs-...


In [15]:
# =========================================================
# FINAL PUBLIC-RECORD EVIDENCE
# =========================================================

new_records = [
    {
        "year": "Current website",
        "source": "South Seattle Student Assessed Fees",
        "topic": "International student S&A exemption",
        "finding": (
            "South Seattle explicitly lists International "
            "Programs among S&A-exempt populations. "
            "This does not independently establish "
            "Seattle Central's assessment practices."
        ),
        "status": "Verified sister-college policy",
        "source_url": (
            "https://southseattle.edu/student-life/"
            "student-leadership/student-assessed-fees"
        ),
    },
    {
        "year": "2023",
        "source": "Seattle Central ASC Procedural Manual",
        "topic": "Annual S&A revenue estimates",
        "finding": (
            "The Director of Business Operations estimates "
            "S&A income available for allocation. "
            "Estimates do not establish actual collections."
        ),
        "status": "Verified college procedure",
        "source_url": (
            "https://studentleadership.seattlecentral.edu/"
            "sites/studentleadership.seattlecentral.edu/"
            "files/2023-09/procedural-manual-2023.pdf"
        ),
    },
    {
        "year": "FY2025",
        "source": "Seattle Colleges Audited Financial Statements",
        "topic": "Published revenue aggregation",
        "finding": (
            "Seattle Central revenue is presented in broad "
            "categories. Separate S&A and International "
            "Activity Fee collections cannot be determined "
            "from these statements."
        ),
        "status": "Reporting limitation",
        "source_url": (
            "https://www.seattlecolleges.edu/sites/"
            "seattlecolleges.southseattle.edu/files/"
            "2026-03/FY25%20Seattle%20Colleges%20"
            "Audited%20Financial%20Statements.pdf"
        ),
    },
]

LOG_PATH = (
    PROJECT_ROOT / "data" / "manual"
    / "sa_fund_evidence_log.csv"
)

fund_evidence = pd.read_csv(LOG_PATH)

fund_evidence = (
    pd.concat(
        [fund_evidence, pd.DataFrame(new_records)],
        ignore_index=True
    )
    .drop_duplicates(
        subset=["source", "topic"],
        keep="last"
    )
    .reset_index(drop=True)
)

fund_evidence.to_csv(LOG_PATH, index=False)

print(f"Total evidence records: {len(fund_evidence)}")
display(fund_evidence.tail(3))

Total evidence records: 23


,year,source,topic,finding,status,source_url
20,Current website,South Seattle Student Assessed Fees,International student S&A exemption,South Seattle explicitly lists International P...,Verified sister-college policy,https://southseattle.edu/student-life/student-...
21,2023,Seattle Central ASC Procedural Manual,Annual S&A revenue estimates,The Director of Business Operations estimates ...,Verified college procedure,https://studentleadership.seattlecentral.edu/s...
22,FY2025,Seattle Colleges Audited Financial Statements,Published revenue aggregation,Seattle Central revenue is presented in broad ...,Reporting limitation,https://www.seattlecolleges.edu/sites/seattlec...


In [16]:
# =========================================================
# FINAL RESEARCH DATA QUALITY CHECK
# =========================================================

print("FINAL INTERNATIONAL S&A RESEARCH VALIDATION")
print("=" * 55)

MANUAL = PROJECT_ROOT / "data" / "manual"
PROCESSED = PROJECT_ROOT / "data" / "processed"

# Load saved files independently
enrollment_raw = pd.read_csv(
    MANUAL / "international_enrollment_context.csv"
)

enrollment_clean = pd.read_csv(
    PROCESSED / "international_enrollment_summary.csv"
)

evidence = pd.read_csv(
    MANUAL / "sa_fund_evidence_log.csv"
)

questions = pd.read_csv(
    MANUAL / "sa_fund_open_questions.csv"
)

fee_reference = pd.read_csv(
    MANUAL / "international_fee_reference.csv"
)


# ---------------------------------------------------------
# 1. FILE STRUCTURE
# ---------------------------------------------------------

checks = {
    "Enrollment source: 3 rows": len(enrollment_raw) == 3,
    "Processed enrollment: 3 rows": len(enrollment_clean) == 3,
    "Evidence log: 23 records": len(evidence) == 23,
    "Open questions: 4 records": len(questions) == 4,
    "Fee reference: 14 rows": len(fee_reference) == 14,
}


# ---------------------------------------------------------
# 2. ENROLLMENT CONSISTENCY
# ---------------------------------------------------------

years = {"2023-24", "2024-25", "2025-26"}

checks["Academic years correct"] = (
    set(enrollment_clean["academic_year"]) == years
)

checks["No duplicate enrollment years"] = (
    not enrollment_clean["academic_year"].duplicated().any()
)

# Compare all original count columns
count_columns = [
    "total_headcount",
    "international_all_funds",
    "international_state_fund",
    "international_contract_fund",
    "international_contract_program_all",
    "international_contract_program_contract",
]

comparison = enrollment_raw.merge(
    enrollment_clean,
    on="academic_year",
    suffixes=("_raw", "_clean"),
    validate="one_to_one"
)

checks["Original counts preserved"] = (
    len(comparison) == 3
    and all(
        (
            comparison[f"{col}_raw"]
            == comparison[f"{col}_clean"]
        ).all()
        for col in count_columns
    )
)

checks["Contract differences preserved"] = (
    enrollment_clean
    .sort_values("academic_year")[
        "contract_group_difference"
    ]
    .tolist() == [4, 4, 3]
)

expected_status = {
    "2023-24": "final",
    "2024-25": "final",
    "2025-26": "provisional",
}

checks["Reporting statuses correct"] = all(
    row["status"] == expected_status[row["academic_year"]]
    for _, row in enrollment_clean.iterrows()
)


# ---------------------------------------------------------
# 3. EVIDENCE LOG
# ---------------------------------------------------------

checks["No duplicate evidence entries"] = (
    not evidence[
        ["source", "topic"]
    ].duplicated().any()
)

checks["Evidence sources documented"] = (
    evidence[
        ["source", "topic", "finding", "status", "source_url"]
    ]
    .notna()
    .all()
    .all()
)


# ---------------------------------------------------------
# 4. OPEN RESEARCH QUESTIONS
# ---------------------------------------------------------

checks["No duplicate research questions"] = (
    not questions["question"].duplicated().any()
)

checks["Question statuses documented"] = (
    questions["status"].notna().all()
)


# ---------------------------------------------------------
# 5. INTERNATIONAL FEES
# ---------------------------------------------------------

checks["No duplicate fee entries"] = (
    not fee_reference[
        ["source_snapshot", "fee_name"]
    ].duplicated().any()
)

checks["Both fee snapshots present"] = (
    set(fee_reference["source_snapshot"])
    == {"2025_board_packet", "current_website"}
)

# Missing amounts are permitted only for variable fees.
checks["Variable fee amounts handled correctly"] = (
    fee_reference.loc[
        fee_reference["published_amount_usd"].isna(),
        "amount_type"
    ].eq("variable").all()
    and
    fee_reference.loc[
        fee_reference["amount_type"] == "fixed",
        "published_amount_usd"
    ].notna().all()
)


# ---------------------------------------------------------
# FINAL RESULTS
# ---------------------------------------------------------

qa_results = pd.DataFrame(
    [
        {"check": name, "passed": bool(result)}
        for name, result in checks.items()
    ]
)

display(qa_results)

print("=" * 55)

if qa_results["passed"].all():
    print("ALL RESEARCH DATA VALIDATED")
else:
    print("VALIDATION FAILED")
    display(qa_results[~qa_results["passed"]])

FINAL INTERNATIONAL S&A RESEARCH VALIDATION


,check,passed
0,Enrollment source: 3 rows,True
1,Processed enrollment: 3 rows,True
2,Evidence log: 23 records,True
3,Open questions: 4 records,True
4,Fee reference: 14 rows,True
5,Academic years correct,True
6,No duplicate enrollment years,True
7,Original counts preserved,True
8,Contract differences preserved,True
9,Reporting statuses correct,True


ALL RESEARCH DATA VALIDATED
